In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_vit
from train import WarmUpCosine, AdamW, LastNSaver, make_train_ds, make_test_ds

In [4]:
initial_lr = 2e-4
weight_decay = 1e-5
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [5]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [6]:
model = build_vit(num_classes=4)

In [7]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay
)

In [8]:
loss_fn = tf.keras.losses.CategoricalCrossentropy()
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

In [9]:
saver = LastNSaver(n=20)

In [10]:
train_ds = make_train_ds(
    x_train, y_train_onehot,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.2   # 0.2~0.4 都常用；过拟合明显就加大
)
test_ds = make_test_ds(x_test, y_test_onehot, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 21s - loss: 1.2912 - accuracy: 0.3851 - val_loss: 2.0004 - val_accuracy: 0.2110 - 21s/epoch - 135ms/step
Epoch 2/200
156/156 - 11s - loss: 1.1400 - accuracy: 0.5057 - val_loss: 2.1288 - val_accuracy: 0.3097 - 11s/epoch - 68ms/step
Epoch 3/200
156/156 - 11s - loss: 1.0850 - accuracy: 0.5436 - val_loss: 1.2697 - val_accuracy: 0.4383 - 11s/epoch - 68ms/step
Epoch 4/200
156/156 - 11s - loss: 1.0364 - accuracy: 0.5767 - val_loss: 0.9474 - val_accuracy: 0.5960 - 11s/epoch - 69ms/step
Epoch 5/200
156/156 - 11s - loss: 1.0115 - accuracy: 0.5979 - val_loss: 0.8542 - val_accuracy: 0.6503 - 11s/epoch - 68ms/step
Epoch 6/200
156/156 - 11s - loss: 0.9679 - accuracy: 0.6235 - val_loss: 0.9764 - val_accuracy: 0.5867 - 11s/epoch - 68ms/step
Epoch 7/200
156/156 - 10s - loss: 0.9316 - accuracy: 0.6474 - val_loss: 0.8047 - val_accuracy: 0.6783 - 10s/epoch - 67ms/step
Epoch 8/200
156/156 - 10s - loss: 0.8944 - accuracy: 0.6671 - val_loss: 0.7389 - val_accuracy: 0.7105 - 10s/epoch - 6

In [11]:
model.save("ViT_8.h5")